# Retinal Vascular Graph Stability Analysis

Topology and geometry sensitivity using the official HRF and CHASE_DB1 retinal vessel datasets.

In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from retina_graph.data import ensure_hrf_data, ensure_chase_data, iter_all_samples
from retina_graph.graph_features import perturbation_set, extract_graph_features
from retina_graph.experiment import run_graph_experiment
from skimage.morphology import skeletonize

HRF_ROOT = ensure_hrf_data(ROOT)
INCLUDE_CHASE = os.environ.get("INCLUDE_CHASE", "0") == "1"
CHASE_ROOT = ensure_chase_data(ROOT) if INCLUDE_CHASE else None
RESULTS = ROOT / "results"

## Inspect an expert retinal vessel graph

In [ ]:
sample = next(iter_all_samples(HRF_ROOT, CHASE_ROOT, include_chase=False))
metrics, graph, branches = extract_graph_features(sample.vessel_mask, sample.fov_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(sample.image)
axes[0].set_title("Original fundus image")
axes[1].imshow(sample.vessel_mask, cmap="gray")
axes[1].set_title("Expert vessel annotation")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

skeleton = skeletonize(sample.vessel_mask & sample.fov_mask)
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(sample.image)
ax.imshow(__import__("numpy").ma.masked_where(~skeleton, skeleton), cmap="autumn", alpha=0.75)
for node_id, attributes in graph.nodes(data=True):
    marker = "s" if attributes["node_type"] == "junction" else "o"
    ax.scatter(attributes["x"], attributes["y"], marker=marker, s=12)
ax.set_title("Skeleton and contracted vascular graph nodes")
ax.axis("off")
plt.tight_layout()
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(RESULTS / "figures" / "example_graph_overlay.png", dpi=180)

pd.Series(metrics).to_frame("value")

## Extract branch and graph measurements under controlled perturbations

In [ ]:
per_image, branch_features, summary = run_graph_experiment(
    HRF_ROOT,
    RESULTS,
    chase_root=CHASE_ROOT,
    include_chase=INCLUDE_CHASE,
    sample_limit=int(os.environ.get("RETINA_SAMPLE_LIMIT", "0")) or None,
    prune_length_px=5.0,
)
summary

## Compare sensitivity across datasets

In [ ]:
display(per_image.groupby(["dataset", "condition"])[["branch_count", "endpoint_count", "junction_count", "total_branch_length_px", "mean_tortuosity", "median_vessel_diameter_px"]].mean())
display(branch_features.head(10))

from IPython.display import Image, display
for dataset in summary["dataset"].drop_duplicates():
    display(Image(filename=str(RESULTS / "figures" / f"{dataset.lower()}_graph_stability.png")))

## Reproducibility and interpretation

GraphML files for original masks and the per-image and per-branch tables are saved under `results/`. Relative changes are summarized separately for HRF and CHASE_DB1 because their image resolutions and field-of-view handling differ. The first CHASE_DB1 human annotation is used consistently.